In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

project_path = "/content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/object segmentation"
os.makedirs(project_path, exist_ok=True)

In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="VCLkIHPyGp4Yk2L3u2Ef")
project = rf.workspace("segmentation-dw9lq").project("car-segmentation-lqmse")
version = project.version(3)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 15.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Car-Segmentation-3 in yolov8:: 100%|██████████| 13924/13924 [00:03<00:00, 4563.44it/s]


In [4]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.8 MB/s eta 0:00:00


In [5]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")  # lightweight
# or:
# model = YOLO("yolov8s-seg.pt")  # better accuracy

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,           # GPU in Colab
    verbose=True,
    project=project_path,
    name="training",
    pretrained=True
)

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Car-Segmentation-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=training, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

In [7]:
metrics_val = model.val(
    data=f"{dataset.location}/data.yaml",
    split="val",
    project=project_path,
    name="validation",
    verbose=True
)

print("Validation Box mAP50:", metrics_val.box.map50)
print("Validation Mask mAP50:", metrics_val.seg.map50)
print("Validation Mask mAP50-95:", metrics_val.seg.map)

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,260,989 parameters, 0 gradients, 11.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1933.1±411.7 MB/s, size: 114.7 KB)
val: Scanning /content/Car-Segmentation-3/valid/labels.cache... 1048 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1048/1048 293.0Mit/s 0.0s
val: /content/Car-Segmentation-3/valid/images/1720689_jpg.rf.c9a079339bc71765a5efc7e384ca3a6a.jpg: 1 duplicate labels removed
val: /content/Car-Segmentation-3/valid/images/1722712_jpg.rf.9a5f0b0813f035f30cd54e5af8a69566.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 66/66 3.5it/s 19.1s
                   all       1048       2587      0.752      0.906      0.848      0.795       0.68      0.832      0.756      0.675
               backcar         51        

In [8]:
metrics_test = model.val(
    data=f"{dataset.location}/data.yaml",
    split="test",
    project=project_path,
    name="testing",
    verbose=True
)

print("Test Box mAP50:", metrics_test.box.map50)
print("Test Mask mAP50:", metrics_test.seg.map50)
print("Test Mask mAP50-95:", metrics_test.seg.map)
print("Test Precision (Mask):", metrics_test.seg.p)
print("Test Recall (Mask):", metrics_test.seg.r)

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 27.3±11.1 MB/s, size: 110.9 KB)
val: Scanning /content/Car-Segmentation-3/test/labels... 345 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 345/345 495.6it/s 0.7s
val: New cache created: /content/Car-Segmentation-3/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 2.2it/s 10.0s
                   all        345        789      0.754      0.875      0.845      0.799      0.682      0.803      0.758       0.69
               backcar         21         21      0.965          1      0.995      0.978      0.965          1      0.995      0.987
          backcarright         20         20      0.673        0.8        0.8        0.8      0.659        0.8        0.8      0.795
           backleftcar         20         20      0.502   

In [9]:
results = model.predict(
    source=f"{project_path}/image.jpg",
    conf=0.25,
    device=0,
    save=True,
    project=project_path,
    name="predictions"
)


image 1/1 /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/object segmentation/image.jpg: 448x640 1 exteriorcar, 1 frontleftcar, 1 frontrightcar, 55.6ms
Speed: 3.1ms preprocess, 55.6ms inference, 17.9ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/drive/MyDrive/Colab Notebooks/computer vision/Ultralytics YOLO/object segmentation/predictions


In [10]:
# model = YOLO(f"{project_path}/training/weights/best.pt")

# results = model("test_image.jpg", conf=0.25, save=True)